# 02 - 数据集与标注格式

**学习目标：**
- 理解 YOLO 标注格式：`class_id x_center y_center width height`
- 理解归一化坐标 vs 像素坐标
- 掌握数据集的目录组织方式
- 学会使用 `data.yaml` 配置数据集
- 分析 COCO128 数据集的结构和统计信息

---

## 1. YOLO 标注格式详解

YOLO 使用纯文本文件存储标注，每行一个目标：

```
class_id  x_center  y_center  width  height
```

关键点：
- **class_id**: 类别的数字编号（从 0 开始）
- **x_center, y_center**: 框中心点的坐标（**归一化到 0-1**）
- **width, height**: 框的宽高（**归一化到 0-1**）

```
图片尺寸: 640 × 480
┌────────────────────────────────┐
│ (0,0)                  (640,0) │
│         ┌──────────┐          │
│         │  目标框   │          │
│         │ (320,240) │          │  ← 中心点 (320, 240)
│         │  200×150  │          │  ← 宽 200, 高 150
│         └──────────┘          │
│ (0,480)                (640,480)│
└────────────────────────────────┘

YOLO 标注:
  x_center = 320 / 640 = 0.5
  y_center = 240 / 480 = 0.5
  width    = 200 / 640 = 0.3125
  height   = 150 / 480 = 0.3125

→ 0 0.5 0.5 0.3125 0.3125
```

**为什么要归一化？**
- 不同图片尺寸不同，归一化后坐标统一在 0-1 范围
- 模型更容易学习相对位置而非绝对像素
- 方便进行数据增强（缩放、裁剪等）

## 2. 数据集目录结构

YOLO 数据集的标准组织方式：

```
dataset/
├── images/
│   ├── train/
│   │   ├── 000001.jpg
│   │   ├── 000002.jpg
│   │   └── ...
│   └── val/
│       ├── 000101.jpg
│       └── ...
├── labels/
│   ├── train/
│   │   ├── 000001.txt  ← 与图片同名，扩展名 .txt
│   │   ├── 000002.txt
│   │   └── ...
│   └── val/
│       ├── 000101.txt
│       └── ...
└── data.yaml           ← 数据集配置文件
```

**关键约定**：
- 图片和标注文件**同名**（仅扩展名不同）
- 标注文件在 `labels/` 目录，图片在 `images/` 目录
- 训练集和验证集分开

## 3. 动手实验：加载 COCO128 数据集

ultralytics 内置了 COCO128 数据集，会自动下载。

In [ ]:
from pathlib import Path

# COCO128 数据集路径（使用我们已经下载到 data/ 目录的本地数据）
coco128_dir = Path("../data/coco128")
print(f"COCO128 数据集目录: {coco128_dir.resolve()}")
print(f"是否存在: {coco128_dir.exists()}")

# 如果不存在，提示运行下载脚本
if not coco128_dir.exists():
    print("数据集不存在，请先运行: uv run python scripts/download_dataset.py")
    raise FileNotFoundError("数据集未下载")

In [ ]:
# 探索目录结构
images_dir = coco128_dir / "images" / "train2017"
labels_dir = coco128_dir / "labels" / "train2017"

image_files = sorted(list(images_dir.glob("*.jpg")))
label_files = sorted(list(labels_dir.glob("*.txt")))

print(f"图片数量: {len(image_files)}")
print(f"标注数量: {len(label_files)}")
print(f"\n前 5 张图片: {[f.name for f in image_files[:5]]}")
print(f"前 5 个标注: {[f.name for f in label_files[:5]]}")

## 4. 查看标注文件内容

In [ ]:
# 读取第一个标注文件
first_label = label_files[0]
print(f"标注文件: {first_label.name}")
print(f"对应图片: {first_label.stem}.jpg\n")

with open(first_label) as f:
    lines = f.readlines()

print(f"标注行数（目标数量）: {len(lines)}")
print(f"\n标注内容:")
for i, line in enumerate(lines):
    parts = line.strip().split()
    class_id = int(parts[0])
    x_center, y_center, w, h = [float(x) for x in parts[1:5]]
    print(f"  目标 {i+1}: class={class_id}, center=({x_center:.3f}, {y_center:.3f}), size=({w:.3f}, {h:.3f})")

## 5. data.yaml 配置文件

数据集的"身份证"，告诉模型数据在哪里、有哪些类别。

In [ ]:
import yaml

# 读取我们自己的 data.yaml
data_yaml = Path("../configs/data/coco128.yaml")
with open(data_yaml) as f:
    config = yaml.safe_load(f)

print("data.yaml 内容:")
print(f"  path: {config.get('path')}")
print(f"  train: {config.get('train')}")
print(f"  val: {config.get('val')}")
print(f"  nc (类别数): {config.get('nc')}")
print(f"  names (前 10 个类别): {dict(list(config.get('names', {}).items())[:10])}")

## 6. 数据集统计分析

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# 统计每张图片的目标数量
boxes_per_image = []
class_counts = Counter()

for label_file in label_files:
    with open(label_file) as f:
        lines = [l.strip() for l in f if l.strip()]
        boxes_per_image.append(len(lines))
        for line in lines:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

print(f"每张图片的目标数:")
print(f"  最少: {min(boxes_per_image)}")
print(f"  最多: {max(boxes_per_image)}")
print(f"  平均: {sum(boxes_per_image)/len(boxes_per_image):.1f}")
print(f"  总标注框数: {sum(boxes_per_image)}")

In [ ]:
# 可视化：每张图片的目标数量分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：目标数量分布
axes[0].hist(boxes_per_image, bins=range(0, max(boxes_per_image)+2), edgecolor='black')
axes[0].set_xlabel('每张图片的目标数量')
axes[0].set_ylabel('图片数量')
axes[0].set_title('目标数量分布')

# 右图：类别出现频率（前 20 个类别）
top_classes = class_counts.most_common(20)
class_names = [config['names'][c[0]] for c in top_classes]
counts = [c[1] for c in top_classes]

axes[1].barh(range(len(class_names)), counts)
axes[1].set_yticks(range(len(class_names)))
axes[1].set_yticklabels(class_names)
axes[1].set_xlabel('出现次数')
axes[1].set_title('类别频率 Top 20')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 7. 可视化标注

In [ ]:
from PIL import Image, ImageDraw

# 读取一张图片及其标注
img_path = image_files[0]
label_path = label_files[0]

img = Image.open(img_path)
w, h = img.size
print(f"图片: {img_path.name}, 尺寸: {w}×{h}")

# 读取标注
draw = ImageDraw.Draw(img)
colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange']

with open(label_path) as f:
    for i, line in enumerate(f):
        parts = line.strip().split()
        cls = int(parts[0])
        cx, cy, bw, bh = [float(x) for x in parts[1:5]]
        
        # 转换为像素坐标
        x1 = (cx - bw/2) * w
        y1 = (cy - bh/2) * h
        x2 = (cx + bw/2) * w
        y2 = (cy + bh/2) * h
        
        color = colors[i % len(colors)]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        
        class_name = config['names'].get(cls, str(cls))
        draw.text((x1, y1-15), f"{class_name}", fill=color)
        
        print(f"  目标 {i+1}: {class_name} at ({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})")

img

## 8. 坐标格式转换

YOLO 用归一化坐标，但实际中你可能遇到多种格式：

In [ ]:
# 常见的标注格式对比
print("=" * 60)
print("标注格式对比")
print("=" * 60)

# 假设图片尺寸 640×480，目标框在 (120, 80) 到 (520, 400)
img_w, img_h = 640, 480
x1, y1, x2, y2 = 120, 80, 520, 400

print(f"\n图片尺寸: {img_w}×{img_h}")
print(f"目标框像素坐标: ({x1}, {y1}, {x2}, {y2})")

# 1. XYXY 格式 (Pascal VOC, COCO)
print(f"\n1. XYXY 格式 (像素坐标):")
print(f"   {x1} {y1} {x2} {y2}")

# 2. XYWH 格式 (COCO bbox)
bw = x2 - x1
bh = y2 - y1
print(f"\n2. XYWH 格式 (左上角 + 宽高):")
print(f"   {x1} {y1} {bw} {bh}")

# 3. YOLO 格式 (归一化的中心点 + 宽高)
cx = (x1 + x2) / 2 / img_w
cy = (y1 + y2) / 2 / img_h
nw = bw / img_w
nh = bh / img_h
print(f"\n3. YOLO 格式 (归一化):")
print(f"   0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
print(f"   ↑ class_id")

# 验证转换
print(f"\n验证: 还原回像素坐标")
rx1 = (cx - nw/2) * img_w
ry1 = (cy - nh/2) * img_h
rx2 = (cx + nw/2) * img_w
ry2 = (cy + nh/2) * img_h
print(f"   ({rx1:.0f}, {ry1:.0f}, {rx2:.0f}, {ry2:.0f})  ✓ 与原始坐标一致")

## 9. 练习

### 练习 1: 手动创建标注
找一张图片，手动测量目标的像素坐标，转换为 YOLO 格式写入 .txt 文件。

### 练习 2: 分析标注质量
检查 COCO128 中是否有异常标注（如框超出图片边界、面积为 0 等）。

### 练习 3: 类别分布
统计 COCO128 中各类别的出现频率，哪些类别最多？哪些最少？

## 📝 学习检查

加载本章检查点和练习，检验学习效果：

```python
from yolo_learn.pedagogy.checkpoint import Quiz
from yolo_learn.pedagogy.scaffold import ExerciseSet
from yolo_learn.pedagogy.reflection import ReflectionSet, MistakeLibrary
```


In [ ]:
# 加载本章检查点
quiz = Quiz.from_yaml("02_dataset_and_annotations")
print(f"本章 {quiz.total} 个检查点 + 预测练习")

# 查看第一题
print(quiz.ask(0))


### ✍️ 练习


In [ ]:
# 加载本章练习
exercises = ExerciseSet.from_yaml("02_dataset_and_annotations")
print(f"本章 {exercises.total} 个练习")
for ex in exercises.exercises:
    print()
    print(ex.show())


### 🤔 反思与自评


In [ ]:
# 加载反思问题和自评量表
reflections = ReflectionSet.from_yaml("02_dataset_and_annotations")
print(reflections.show_reflections())
print()
print(reflections.show_assessments())

# 常见错误
mistakes = MistakeLibrary()
print()
print(mistakes.show_for_topic("dataset_and_annotations"))


---

**上一节**: [01 - 什么是目标检测](01_what_is_object_detection.ipynb)

**下一节**: [03 - YOLO 架构](03_yolo_architecture.ipynb) - 理解 YOLO 的网络结构